# Classification — Hands-on

**Companion to deck 06 (Classification with sklearn).** Push past the LogReg baseline with feature engineering, scaling, and tree models.

<a href="https://colab.research.google.com/github/Petkub/MachineLearningLab/blob/main/colab_exercises/06_classification.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

df = sns.load_dataset('titanic').rename(columns={
    'pclass': 'Pclass', 'sex': 'Sex', 'age': 'Age',
    'sibsp': 'SibSp', 'parch': 'Parch',
    'fare': 'Fare', 'survived': 'Survived', 'embarked': 'Embarked',
})
df = df[['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'Survived']].copy()
print(df.shape, '\n')
df.head()

---
## Problem 01 — encode categoricals + impute

Models eat numbers. Strings break sklearn.

Tasks:
1. Map `Sex` to 0/1.
2. Fill missing `Age` with the mean.
3. Fill missing `Fare` with the median.
4. Map `Embarked` to integers (use `pd.factorize` for a quick label encode); fill any NaN with 0 first.
5. Confirm `df.isnull().sum().sum() == 0`.

In [ ]:
# TODO
df['Sex']      = ...
df['Age']      = ...
df['Fare']     = ...
df['Embarked'] = df['Embarked'].fillna('S')
df['Embarked'] = ...   # turn into integer codes

print('total NaN remaining:', df.isnull().sum().sum())
print(df.dtypes)

In [ ]:
assert df.isnull().sum().sum() == 0, 'NaN still present'
assert df['Sex'].dtype != object, 'Sex still string — needs encoding'
assert df['Embarked'].dtype != object, 'Embarked still string — needs encoding'
print('Q1 ok')

<details><summary>Hint</summary>

`df['Sex'].map({'male':0,'female':1})`. `df['Age'].fillna(df['Age'].mean())`. `pd.factorize(df['Embarked'])[0]` returns integer codes.
</details>

---
## Problem 02 — feature engineering

Add 2 new features that often help on tabular tasks.

Tasks:
1. `FamilySize = SibSp + Parch + 1`
2. `IsAlone = (FamilySize == 1).astype(int)`
3. `FarePerPerson = Fare / FamilySize`
4. Confirm all 3 columns exist + no NaN.

In [ ]:
# TODO
df['FamilySize']    = ...
df['IsAlone']       = ...
df['FarePerPerson'] = ...

df.head()

In [ ]:
for c in ['FamilySize', 'IsAlone', 'FarePerPerson']:
    assert c in df.columns, f'missing {c}'
    assert df[c].isnull().sum() == 0, f'{c} has NaN'
print('Q2 ok')

---
## Problem 03 — split, then run 4 models

Tasks:
1. `X` = all columns except `Survived`. `y` = `Survived`.
2. 80/20 split, stratified, random_state=42.
3. Train: LogReg, RF, GradBoost, KNN. Score on test.
4. Save accuracies in `scores` dict.

In [ ]:
# TODO
X = ...
y = ...
X_train, X_test, y_train, y_test = ...

scores = {}
# train + score each model

for name, s in sorted(scores.items(), key=lambda kv: -kv[1]):
    print(f'{name:25s} {s:.3f}')

In [ ]:
assert set(scores.keys()) >= {'logreg', 'rf', 'gb', 'knn'}
for name, s in scores.items():
    assert 0.6 < s < 0.95, f'{name} weird score: {s}'
print('Q3 ok — leader:', max(scores, key=scores.get))

---
## Problem 04 — KNN gotcha: scale matters

KNN measures Euclidean distance. `Fare` (0–500) dominates `Age` (0–80) which dominates `Sex` (0–1). Scale features to put them on equal footing.

Tasks:
1. Build a `Pipeline` of `StandardScaler` + `KNeighborsClassifier(n_neighbors=5)`.
2. Fit on train, score on test.
3. Compare against the un-scaled KNN from Q3 — by how many points did scaling lift it?

In [ ]:
# TODO
knn_scaled = Pipeline([
    # ('scaler', ...),
    # ('knn',    ...),
])
# fit + score
scaled_acc = ...

print('KNN un-scaled:', round(scores['knn'], 3))
print('KNN scaled:   ', round(scaled_acc, 3))
print('lift:         ', round(scaled_acc - scores['knn'], 3))

In [ ]:
assert scaled_acc >= scores['knn'] - 0.02, 'scaled KNN should be at least as good as un-scaled'
print('Q4 ok')

<details><summary>Hint</summary>

`Pipeline([('scaler', StandardScaler()), ('knn', KNeighborsClassifier(n_neighbors=5))])` then `.fit(X_train, y_train)` and `.score(X_test, y_test)`.
</details>

---
## Problem 05 — feature importances from RandomForest

RF tells you which features mattered. Plot them sorted.

Tasks:
1. Refit a `RandomForestClassifier(n_estimators=200, random_state=42)`.
2. Build a sorted `pd.Series` of importances indexed by feature name.
3. Bar plot, descending.

In [ ]:
# TODO
rf = ...
rf.fit(X_train, y_train)
importances = ...   # pd.Series indexed by feature name, sorted descending

importances.plot.bar(figsize=(8, 4))
plt.title('Feature importances — RandomForest')
plt.tight_layout()
plt.show()

print(importances.head())

<details><summary>Hint</summary>

`pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)`.
</details>

---
## Problem 06 — cross-validation

A single train/test split can mislead you (lucky split). Cross-validation averages over k folds for a more honest score.

Tasks:
1. Run `cross_val_score(rf, X, y, cv=5)`.
2. Report mean and std.

In [ ]:
# TODO
cv_scores = ...

print('per-fold:', np.round(cv_scores, 3))
print(f'mean: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')

In [ ]:
assert len(cv_scores) == 5
assert 0.7 < cv_scores.mean() < 0.95
print('Q6 ok')

---
## Solutions

<details><summary>Show all solutions</summary>

```python
# Q1
df['Sex']      = df['Sex'].map({'male': 0, 'female': 1})
df['Age']      = df['Age'].fillna(df['Age'].mean())
df['Fare']     = df['Fare'].fillna(df['Fare'].median())
df['Embarked'] = df['Embarked'].fillna('S')
df['Embarked'] = pd.factorize(df['Embarked'])[0]

# Q2
df['FamilySize']    = df['SibSp'] + df['Parch'] + 1
df['IsAlone']       = (df['FamilySize'] == 1).astype(int)
df['FarePerPerson'] = df['Fare'] / df['FamilySize']

# Q3
X = df.drop('Survived', axis=1)
y = df['Survived']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

models = {
    'logreg': LogisticRegression(max_iter=1000),
    'rf':     RandomForestClassifier(n_estimators=100, random_state=42),
    'gb':     GradientBoostingClassifier(random_state=42),
    'knn':    KNeighborsClassifier(n_neighbors=5),
}
scores = {}
for name, m in models.items():
    m.fit(X_train, y_train)
    scores[name] = accuracy_score(y_test, m.predict(X_test))

# Q4
knn_scaled = Pipeline([
    ('scaler', StandardScaler()),
    ('knn',    KNeighborsClassifier(n_neighbors=5)),
])
knn_scaled.fit(X_train, y_train)
scaled_acc = knn_scaled.score(X_test, y_test)

# Q5
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
importances = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)

# Q6
cv_scores = cross_val_score(rf, X, y, cv=5)
```
</details>